In [1]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))

In [2]:
from src.transforms import test_transforms
from src.dataset import ImageDataset
import torch

annot_path = Path("../data/preprocessed/trainval/annotations.csv")
img_dir = Path("../data/preprocessed/trainval/images")

# test dataset without transforms
dataset = ImageDataset(annot_path, img_dir)
transformed_dataset = ImageDataset(annot_path, img_dir, test_transforms)

In [3]:
def collate_fn(batch):
    # unpacks the batch, and then zip creates separate tuples for images and targets
    images, targets = zip(*batch)

    images = torch.stack(images)
    targets = list(targets)

    return images, targets

In [4]:
from torch.utils.data import DataLoader

dl = DataLoader(transformed_dataset, batch_size=4, collate_fn=collate_fn)
dl

In [5]:
X_batch, y_batch = next(iter(dl))

X_batch.shape, len(y_batch)

(torch.Size([4, 3, 224, 224]), 4)

In [6]:
from src.utilities import compute_giou, cxcywh_to_xyxy

bbox_preds = torch.randn(32, 100, 4)

In [7]:
_, truth_labels = next(iter(transformed_dataset))
truth_boxes = cxcywh_to_xyxy(truth_labels[:, -4:])
truth_boxes.shape, truth_boxes

(torch.Size([5, 4]),
 tensor([[117.5000, 126.0000, 144.5000, 202.0000],
         [ 74.5000, 157.5000, 113.5000, 222.5000],
         [  2.0000, 146.0000,  30.0000, 224.0000],
         [108.0000, 115.5000, 132.0000, 178.5000],
         [124.0000, 111.0000, 140.0000, 131.0000]]))

In [8]:
giou = compute_giou(truth_boxes, truth_boxes)

tensor([[2.0520e+03, 4.4500e-06, 5.6000e-06, 7.6125e+02, 8.0000e+01],
        [4.4500e-06, 2.5350e+03, 6.5000e-06, 1.1550e+02, 1.0000e-14],
        [5.6000e-06, 6.5000e-06, 2.1840e+03, 3.2500e-06, 1.0000e-14],
        [7.6125e+02, 1.1550e+02, 3.2500e-06, 1.5120e+03, 1.2400e+02],
        [8.0000e+01, 1.0000e-14, 1.0000e-14, 1.2400e+02, 3.2000e+02]])
tensor([[2052.0000, 5070.0000, 4368.0000, 2262.7500,  559.9998],
        [4104.0000, 2535.0000, 4368.0000, 2908.5000,  639.9998],
        [4104.0000, 5070.0000, 2184.0000, 3024.0000,  639.9998],
        [3342.7500, 4954.5000, 4368.0000, 1512.0000,  515.9998],
        [4024.0000, 5070.0000, 4368.0000, 2900.0000,  319.9999]])
tensor([[ 2052.0000,  6755.0000, 13965.0000,  3157.2500,  2456.9998],
        [ 6755.0000,  2535.0000,  8697.0000,  6152.5000,  7303.2495],
        [13965.0000,  8697.0000,  2184.0000, 14105.0000, 15593.9990],
        [ 3157.2500,  6152.5000, 14105.0000,  1512.0000,  2159.9998],
        [ 2456.9998,  7303.2495, 15593.9990

In [12]:
a = torch.tensor([1, 2, 3, 4]).unsqueeze(0)
a.shape

giou = compute_giou(a, a)
giou

tensor([[4.]])
tensor([[4.]])
tensor([[4.]])


tensor([[1.]])